<a href="https://colab.research.google.com/github/Cheponke/Transformation-Aware-Reliability-Modeling-for-Multi-Signal-Synthetic-Image-Attribution/blob/main/notebooks/07_cross_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5-fold cross-validation and paired t-tests

Manuscript section 3.8.1: *"5-fold cross-validation is applied to ensure the model's stability across different data subsets…
a paired t-test (multi-signal vs single-signal F1-score) is used to determine if the performance differences before and after compression
are statistically significant (p < 0.05)."*

All 19,820 images of the cleaned splits are pooled and divided into 5 folds, balanced by class. Near-duplicates are grouped across the
**whole** pool and each group stays in one fold, so no copy of a test image is ever trained on. For each fold, the other four folds are split
into training and validation (15% of each class), ResNet-50 and EfficientNet-B0 are trained and calibrated, the watermark threshold and confidence
are set on that fold's validation images, and the fold's ~3,960 test images go through all 9 conditions and all four systems.

Across the 5 folds this gives every result a mean and a spread, and five paired F1 values per condition for the t-tests.

**Runtime > Change runtime type > GPU** (T4 or L4). Roughly **1 hour per fold, about 5 hours in total.** Everything is saved to Google Drive
(`thesis-data/cv/`) as it finishes, so if Colab disconnects just run sections 1-2 again and continue with the next fold: finished steps are skipped.

## 1. Get the code

In [1]:
%cd /content
!rm -rf repo
!git clone https://github.com/Cheponke/Transformation-Aware-Reliability-Modeling-for-Multi-Signal-Synthetic-Image-Attribution.git repo
%cd /content/repo
!pip install -q -e ".[train,provenance,dev]"
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (training will be very slow)")

/content
Cloning into 'repo'...
remote: Enumerating objects: 479, done.
remote: Counting objects: 100% (243/243), done.
remote: Compressing objects: 100% (180/180), done.
remote: Total 479 (delta 143), reused 95 (delta 53), pack-reused 236 (from 1)
Receiving objects: 100% (479/479), 6.90 MiB | 12.56 MiB/s, done.
Resolving deltas: 100% (243/243), done.
/content/repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for attribution (pyproject.toml) ... done
GPU: Tesla T4


## 2. Restore the cleaned data and link the results folder on Google Drive

In [2]:
from google.colab import drive
import os
drive.mount("/content/drive")
D = "/content/drive/MyDrive/thesis-data"
ZIP = f"{D}/harmonized.zip"
if not os.path.exists(ZIP):
    raise FileNotFoundError(f"{ZIP} not found: run notebook 01c first")
!unzip -qo $ZIP -d .
!wc -l data/harmonized/train.csv data/harmonized/val.csv data/harmonized/test.csv
!mkdir -p $D/cv && rm -rf outputs/cv && ln -s $D/cv outputs/cv

Mounted at /content/drive
  13849 data/harmonized/train.csv
   2981 data/harmonized/val.csv
   2993 data/harmonized/test.csv
  19823 total


## 3. Run the folds
One cell per fold, about an hour each. The first cell also builds the fold assignment (`folds.csv`, a few minutes) and prints the fold sizes.
If the session drops, rerun sections 1-2 and then the fold you were on: it picks up where it stopped.

In [3]:
!python scripts/cross_validate.py --folds 0

folds: grouping near-duplicates among 19820 images (dHash distance <= 5)
folds: 19702 groups, 106 with more than one image (224 images), largest 3
  fold 0: 3964 images (1981 human / 1983 AI)
  fold 1: 3964 images (1981 human / 1983 AI)
  fold 2: 3964 images (1981 human / 1983 AI)
  fold 3: 3964 images (1982 human / 1982 AI)
  fold 4: 3964 images (1982 human / 1982 AI)
== fold 0: 13477 train / 2379 val / 3964 test
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100% 97.8M/97.8M [00:00<00:00, 172MB/s]
  resnet50: training on cuda: 13477 train / 2379 val images
epoch 1: train_loss=0.3304 val_loss=0.2060 val_acc=0.9172
epoch 2: train_loss=0.2069 val_loss=0.1701 val_acc=0.9344
epoch 3: train_loss=0.1454 val_loss=0.1808 val_acc=0.9269
epoch 4: train_loss=0.1231 val_loss=0.1758 val_acc=0.9357
epoch 5: train_loss=0.1051 val_loss=0.2275 val_acc=0.9185
epoch 6: train_loss=0.0758 val_loss=0.1911 val_acc=0.9348
e

In [4]:
!python scripts/cross_validate.py --folds 1

folds: reusing outputs/cv/folds.csv (19820 images)
== fold 1: 13477 train / 2379 val / 3964 test
  resnet50: training on cuda: 13477 train / 2379 val images
epoch 1: train_loss=0.3350 val_loss=0.1772 val_acc=0.9344
epoch 2: train_loss=0.1968 val_loss=0.1744 val_acc=0.9285
epoch 3: train_loss=0.1464 val_loss=0.1812 val_acc=0.9327
epoch 4: train_loss=0.1239 val_loss=0.1942 val_acc=0.9235
epoch 5: train_loss=0.0967 val_loss=0.2088 val_acc=0.9311
epoch 6: train_loss=0.0818 val_loss=0.2081 val_acc=0.9378
epoch 7: train_loss=0.0784 val_loss=0.2147 val_acc=0.9336
early stop: no val improvement for 5 epochs
  resnet50: temperature=0.901
  efficientnet_b0: training on cuda: 13477 train / 2379 val images
epoch 1: train_loss=0.3626 val_loss=0.2081 val_acc=0.9193
epoch 2: train_loss=0.2244 val_loss=0.1660 val_acc=0.9369
epoch 3: train_loss=0.1689 val_loss=0.1502 val_acc=0.9449
epoch 4: train_loss=0.1359 val_loss=0.1634 val_acc=0.9369
epoch 5: train_loss=0.1190 val_loss=0.1570 val_acc=0.9420
epoch 

In [5]:
!python scripts/cross_validate.py --folds 2

folds: reusing outputs/cv/folds.csv (19820 images)
== fold 2: 13477 train / 2379 val / 3964 test
  resnet50: training on cuda: 13477 train / 2379 val images
epoch 1: train_loss=0.3310 val_loss=0.2124 val_acc=0.9138
epoch 2: train_loss=0.1987 val_loss=0.1841 val_acc=0.9277
epoch 3: train_loss=0.1572 val_loss=0.2913 val_acc=0.9037
epoch 4: train_loss=0.1226 val_loss=0.2135 val_acc=0.9306
epoch 5: train_loss=0.1004 val_loss=0.2214 val_acc=0.9197
epoch 6: train_loss=0.0854 val_loss=0.2422 val_acc=0.9235
epoch 7: train_loss=0.0814 val_loss=0.2411 val_acc=0.9256
early stop: no val improvement for 5 epochs
  resnet50: temperature=1.178
  efficientnet_b0: training on cuda: 13477 train / 2379 val images
epoch 1: train_loss=0.3611 val_loss=0.2252 val_acc=0.9084
epoch 2: train_loss=0.2174 val_loss=0.1865 val_acc=0.9264
epoch 3: train_loss=0.1771 val_loss=0.1825 val_acc=0.9302
epoch 4: train_loss=0.1319 val_loss=0.1878 val_acc=0.9281
epoch 5: train_loss=0.1113 val_loss=0.1861 val_acc=0.9357
epoch 

In [6]:
!python scripts/cross_validate.py --folds 3

folds: reusing outputs/cv/folds.csv (19820 images)
== fold 3: 13477 train / 2379 val / 3964 test
  resnet50: training on cuda: 13477 train / 2379 val images
epoch 1: train_loss=0.3296 val_loss=0.2062 val_acc=0.9142
epoch 2: train_loss=0.1946 val_loss=0.1987 val_acc=0.9281
epoch 3: train_loss=0.1524 val_loss=0.1975 val_acc=0.9285
epoch 4: train_loss=0.1205 val_loss=0.2146 val_acc=0.9264
epoch 5: train_loss=0.0991 val_loss=0.2200 val_acc=0.9277
epoch 6: train_loss=0.0849 val_loss=0.2228 val_acc=0.9231
epoch 7: train_loss=0.0666 val_loss=0.2288 val_acc=0.9374
epoch 8: train_loss=0.0602 val_loss=0.2262 val_acc=0.9378
early stop: no val improvement for 5 epochs
  resnet50: temperature=1.341
  efficientnet_b0: training on cuda: 13477 train / 2379 val images
epoch 1: train_loss=0.3613 val_loss=0.2248 val_acc=0.9058
epoch 2: train_loss=0.2187 val_loss=0.1938 val_acc=0.9210
epoch 3: train_loss=0.1673 val_loss=0.1757 val_acc=0.9264
epoch 4: train_loss=0.1352 val_loss=0.1874 val_acc=0.9248
epoch 

In [7]:
!python scripts/cross_validate.py --folds 4

folds: reusing outputs/cv/folds.csv (19820 images)
== fold 4: 13477 train / 2379 val / 3964 test
  resnet50: training on cuda: 13477 train / 2379 val images
epoch 1: train_loss=0.3194 val_loss=0.2165 val_acc=0.9109
epoch 2: train_loss=0.1882 val_loss=0.2269 val_acc=0.9218
epoch 3: train_loss=0.1509 val_loss=0.1955 val_acc=0.9235
epoch 4: train_loss=0.1120 val_loss=0.2170 val_acc=0.9210
epoch 5: train_loss=0.0916 val_loss=0.2023 val_acc=0.9344
epoch 6: train_loss=0.0796 val_loss=0.2585 val_acc=0.9269
epoch 7: train_loss=0.0743 val_loss=0.2981 val_acc=0.8962
epoch 8: train_loss=0.0704 val_loss=0.2140 val_acc=0.9290
early stop: no val improvement for 5 epochs
  resnet50: temperature=1.297
  efficientnet_b0: training on cuda: 13477 train / 2379 val images
epoch 1: train_loss=0.3576 val_loss=0.2269 val_acc=0.9096
epoch 2: train_loss=0.2081 val_loss=0.1858 val_acc=0.9239
epoch 3: train_loss=0.1639 val_loss=0.1689 val_acc=0.9315
epoch 4: train_loss=0.1274 val_loss=0.1798 val_acc=0.9327
epoch 

## 4. Results across the 5 folds
**mean ± sd** over the folds for each system and condition. The t-tests compare multi-signal F1 with each single-signal F1, pairing the
five folds (each fold's systems saw exactly the same test images). **p** is the paired t-test the manuscript specifies;
**p (Holm)** additionally corrects for running many tests at once (2 backbones × 9 conditions × 3 comparisons), which a panel may ask about.
A system that never flags an AI image (provenance after re-encoding) has F1 = 0. Where the difference is identical in every fold
(e.g. both systems perfect on clean images) the t-test is undefined and marked as such.

In [8]:
!python scripts/cross_validate.py --aggregate-only
import json, pandas as pd
pd.set_option("display.width", 220)
ms = pd.read_csv("outputs/cv/summary_mean_sd.csv")
order = ms.condition.unique()
fmt = lambda m, s: "-" if pd.isna(m) else (f"{m:.3f}" if pd.isna(s) else f"{m:.3f} ± {s:.3f}")
for arch in ms.arch.unique():
    for metric in ["f1", "accuracy", "recall", "fpr", "coverage", "ece"]:
        t = ms[ms.arch == arch].assign(v=lambda d: [fmt(m, s) for m, s in zip(d[f"{metric}_mean"], d[f"{metric}_sd"])])
        print(f"\n{arch} | {metric} (mean ± sd over folds)")
        display(t.pivot(index="condition", columns="system", values="v").loc[order, ["provenance_only", "watermark_only", "cnn_only", "multi_signal"]])

aggregated 5 folds ([0, 1, 2, 3, 4]); 54 paired t-tests, 50 significant at p<0.05 (40 after Holm correction, 2 undefined)

resnet50 | f1 (mean ± sd over folds)


system,provenance_only,watermark_only,cnn_only,multi_signal
condition,,,,
clean,1.000 ± 0.000,1.000 ± 0.000,0.815 ± 0.019,1.000 ± 0.000
jpeg_q90,-,1.000 ± 0.000,0.796 ± 0.020,0.972 ± 0.010
jpeg_q70,-,1.000 ± 0.000,0.796 ± 0.020,0.968 ± 0.015
jpeg_q50,-,1.000 ± 0.000,0.785 ± 0.023,0.956 ± 0.018
jpeg_q30,-,0.999 ± 0.000,0.752 ± 0.025,0.941 ± 0.024
webp_q80,-,1.000 ± 0.000,0.771 ± 0.017,0.981 ± 0.005
resize_0.5,-,1.000 ± 0.000,0.656 ± 0.065,0.986 ± 0.005
resize_0.25,-,0.519 ± 0.016,0.382 ± 0.130,0.644 ± 0.046
recompress_x3,-,1.000 ± 0.000,0.795 ± 0.023,0.977 ± 0.011



resnet50 | accuracy (mean ± sd over folds)


system,provenance_only,watermark_only,cnn_only,multi_signal
condition,,,,
clean,1.000 ± 0.000,1.000 ± 0.000,0.834 ± 0.014,1.000 ± 0.000
jpeg_q90,-,1.000 ± 0.000,0.816 ± 0.014,0.970 ± 0.012
jpeg_q70,-,1.000 ± 0.000,0.814 ± 0.013,0.966 ± 0.018
jpeg_q50,-,1.000 ± 0.000,0.801 ± 0.014,0.952 ± 0.023
jpeg_q30,-,1.000 ± 0.000,0.768 ± 0.016,0.933 ± 0.031
webp_q80,-,1.000 ± 0.000,0.802 ± 0.009,0.980 ± 0.006
resize_0.5,-,1.000 ± 0.000,0.735 ± 0.034,0.986 ± 0.006
resize_0.25,-,0.974 ± 0.007,0.616 ± 0.049,0.758 ± 0.035
recompress_x3,-,1.000 ± 0.000,0.817 ± 0.015,0.975 ± 0.013



resnet50 | recall (mean ± sd over folds)


...

system,provenance_only,watermark_only,cnn_only,multi_signal
condition,,,,
clean,0.000 ± 0.000,0.000 ± 0.000,0.055 ± 0.012,0.000 ± 0.000
jpeg_q90,0.000 ± 0.000,0.000 ± 0.000,0.066 ± 0.013,0.044 ± 0.010
jpeg_q70,0.000 ± 0.000,0.000 ± 0.000,0.051 ± 0.011,0.033 ± 0.010
jpeg_q50,0.000 ± 0.000,0.000 ± 0.000,0.052 ± 0.011,0.035 ± 0.008
jpeg_q30,0.000 ± 0.000,0.000 ± 0.000,0.053 ± 0.010,0.034 ± 0.008
webp_q80,0.000 ± 0.000,0.000 ± 0.000,0.046 ± 0.013,0.028 ± 0.009
resize_0.5,0.000 ± 0.000,0.000 ± 0.000,0.029 ± 0.015,0.017 ± 0.009
resize_0.25,0.000 ± 0.000,0.010 ± 0.003,0.018 ± 0.012,0.018 ± 0.005
recompress_x3,0.000 ± 0.000,0.000 ± 0.000,0.042 ± 0.013,0.026 ± 0.010



efficientnet_b0 | coverage (mean ± sd over folds)


system,provenance_only,watermark_only,cnn_only,multi_signal
condition,,,,
clean,1.000 ± 0.000,0.500 ± 0.000,1.000 ± 0.000,1.000 ± 0.000
jpeg_q90,0.000 ± 0.000,0.500 ± 0.000,1.000 ± 0.000,0.973 ± 0.005
jpeg_q70,0.000 ± 0.000,0.500 ± 0.000,1.000 ± 0.000,0.978 ± 0.005
jpeg_q50,0.000 ± 0.000,0.500 ± 0.000,1.000 ± 0.000,0.978 ± 0.004
jpeg_q30,0.000 ± 0.000,0.499 ± 0.001,1.000 ± 0.000,0.977 ± 0.002
webp_q80,0.000 ± 0.000,0.500 ± 0.000,1.000 ± 0.000,0.977 ± 0.005
resize_0.5,0.000 ± 0.000,0.500 ± 0.000,1.000 ± 0.000,0.985 ± 0.006
resize_0.25,0.000 ± 0.000,0.182 ± 0.008,1.000 ± 0.000,0.937 ± 0.025
recompress_x3,0.000 ± 0.000,0.500 ± 0.000,1.000 ± 0.000,0.980 ± 0.005



efficientnet_b0 | ece (mean ± sd over folds)


system,provenance_only,watermark_only,cnn_only,multi_signal
condition,,,,
clean,0.010 ± 0.000,0.001 ± 0.000,0.116 ± 0.015,0.010 ± 0.000
jpeg_q90,-,0.001 ± 0.000,0.115 ± 0.016,0.006 ± 0.002
jpeg_q70,-,0.001 ± 0.000,0.106 ± 0.017,0.008 ± 0.002
jpeg_q50,-,0.001 ± 0.000,0.119 ± 0.020,0.007 ± 0.002
jpeg_q30,-,0.001 ± 0.000,0.133 ± 0.022,0.008 ± 0.003
webp_q80,-,0.001 ± 0.000,0.139 ± 0.019,0.010 ± 0.002
resize_0.5,-,0.001 ± 0.000,0.270 ± 0.021,0.009 ± 0.002
resize_0.25,-,0.025 ± 0.007,0.265 ± 0.084,0.168 ± 0.055
recompress_x3,-,0.001 ± 0.000,0.108 ± 0.017,0.008 ± 0.002


In [9]:
sig = pd.read_csv("outputs/cv/significance.csv")
print(json.load(open("outputs/cv/cv_report.json")))
cols = ["condition", "comparison", "mean_f1_multi_signal", "mean_f1_single", "mean_difference", "sd_difference",
        "t_statistic", "p_value", "p_holm", "significant_p05", "significant_holm", "note"]
for arch in sig.arch.unique():
    print(f"\n{arch}: paired t-tests, multi-signal vs single-signal F1 (5 folds)")
    display(sig[sig.arch == arch][cols].round(4))

{'folds_done': [0, 1, 2, 3, 4], 'k': 5, 'alpha': 0.05, 'n_tests': 54, 'significant_unadjusted': 50, 'significant_holm': 40, 'undefined_tests': 2}

resnet50: paired t-tests, multi-signal vs single-signal F1 (5 folds)


,condition,comparison,mean_f1_multi_signal,mean_f1_single,mean_difference,sd_difference,t_statistic,p_value,p_holm,significant_p05,significant_holm,note
0,clean,multi_signal vs provenance_only,1.0000,1.0000,0.0000,0.0000,NaN,NaN,NaN,False,False,identical difference in every fold: t-test und...
1,clean,multi_signal vs watermark_only,1.0000,0.9999,0.0001,0.0001,1.6330,0.1778,0.3556,False,False,NaN
2,clean,multi_signal vs cnn_only,1.0000,0.8146,0.1854,0.0192,21.6056,0.0000,0.0009,True,True,NaN
3,jpeg_q90,multi_signal vs provenance_only,0.9718,0.0000,0.9718,0.0100,216.9147,0.0000,0.0000,True,True,NaN
4,jpeg_q90,multi_signal vs watermark_only,0.9718,0.9999,-0.0281,0.0101,-6.2325,0.0034,0.0439,True,True,NaN
5,jpeg_q90,multi_signal vs cnn_only,0.9718,0.7961,0.1757,0.0257,15.3021,0.0001,0.0028,True,True,NaN
6,jpeg_q70,multi_signal vs provenance_only,0.9684,0.0000,0.9684,0.0154,140.2350,0.0000,0.0000,True,True,NaN
7,jpeg_q70,multi_signal vs watermark_only,0.9684,0.9998,-0.0315,0.0155,-4.5358,0.0105,0.0528,True,False,NaN
8,jpeg_q70,multi_signal vs cnn_only,0.9684,0.7957,0.1727,0.0307,12.5629,0.0002,0.0055,True,True,NaN
9,jpeg_q50,multi_signal vs provenance_only,0.9565,0.0000,0.9565,0.0183,116.9926,0.0000,0.0000,True,True,NaN



efficientnet_b0: paired t-tests, multi-signal vs single-signal F1 (5 folds)


,condition,comparison,mean_f1_multi_signal,mean_f1_single,mean_difference,sd_difference,t_statistic,p_value,p_holm,significant_p05,significant_holm,note
27,clean,multi_signal vs provenance_only,1.0000,1.0000,0.0000,0.0000,NaN,NaN,NaN,False,False,identical difference in every fold: t-test und...
28,clean,multi_signal vs watermark_only,1.0000,0.9999,0.0001,0.0001,1.6330,0.1778,0.3556,False,False,NaN
29,clean,multi_signal vs cnn_only,1.0000,0.7311,0.2689,0.0272,22.1386,0.0000,0.0009,True,True,NaN
30,jpeg_q90,multi_signal vs provenance_only,0.9785,0.0000,0.9785,0.0048,455.0185,0.0000,0.0000,True,True,NaN
31,jpeg_q90,multi_signal vs watermark_only,0.9785,0.9999,-0.0214,0.0049,-9.7600,0.0006,0.0123,True,True,NaN
32,jpeg_q90,multi_signal vs cnn_only,0.9785,0.7349,0.2436,0.0282,19.3196,0.0000,0.0011,True,True,NaN
33,jpeg_q70,multi_signal vs provenance_only,0.9837,0.0000,0.9837,0.0048,461.2163,0.0000,0.0000,True,True,NaN
34,jpeg_q70,multi_signal vs watermark_only,0.9837,0.9998,-0.0161,0.0048,-7.4362,0.0017,0.0279,True,True,NaN
35,jpeg_q70,multi_signal vs cnn_only,0.9837,0.7548,0.2289,0.0250,20.4620,0.0000,0.0010,True,True,NaN
36,jpeg_q50,multi_signal vs provenance_only,0.9825,0.0000,0.9825,0.0038,575.9839,0.0000,0.0000,True,True,NaN


## 5. Save it
File > Save a copy in GitHub. The fold assignment, per-fold summaries, mean ± sd table and t-tests are on Drive in `thesis-data/cv/`
and are collected by notebook 06.